# Build Shared LSC Contexts

This notebook creates the shared mention-level context table for the diachronic LSC analyses. It starts from the final quality-gated Common Crawl corpus, re-detects target and comparator mentions, extracts target-centred context windows, and applies the shared cap of up to three mentions per document per analysis unit.

## Setup

The notebook deliberately keeps preprocessing simple and inspectable. It uses the term patterns frozen in `configs/commoncrawl_collection.yaml` so the LSC analysis remains aligned with corpus construction.

In [ ]:
from __future__ import annotations

from collections import defaultdict
from datetime import datetime, timezone
from hashlib import sha1
from pathlib import Path
import re

import pandas as pd
import yaml

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs/commoncrawl_collection.yaml").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate repository root from current working directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "configs/commoncrawl_collection.yaml"
CORPUS_PATH = PROJECT_ROOT / "data/processed/corpus/corpus_documents.parquet"
OUTPUT_DIR = PROJECT_ROOT / "data/interim/lsc/contexts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONTEXT_PATH = OUTPUT_DIR / "lsc_mention_contexts.parquet"
COUNTS_RAW_PATH = OUTPUT_DIR / "lsc_context_counts_by_year_unit_raw_form.csv"
COUNTS_UNIT_PATH = OUTPUT_DIR / "lsc_context_counts_by_year_unit.csv"
TOP_DOMAINS_PATH = OUTPUT_DIR / "lsc_context_top_domains_by_year_unit.csv"
SAMPLES_PATH = OUTPUT_DIR / "lsc_context_manual_samples.csv"
SUMMARY_PATH = OUTPUT_DIR / "lsc_context_extraction_summary.csv"
PUBLICATION_EXCLUSIONS_PATH = OUTPUT_DIR / "lsc_context_publication_date_exclusions.csv"

MENTION_CAP_PER_DOC_UNIT = 3
PUBLICATION_YEAR_START = 2014
PUBLICATION_YEAR_END = 2026
ASD_DISAMBIGUATION_WINDOW_CHARS = 200
LOCAL_SENTENCE_WINDOW_CHARS = 1200
SNIPPET_WINDOW_CHARS = 200
TOKEN_WINDOW_SIZE = 5
ACRONYM_EXPANSION_COLLAPSE_WINDOW_CHARS = 120
RANDOM_SEED = 123

assert CONFIG_PATH.exists(), CONFIG_PATH
assert CORPUS_PATH.exists(), CORPUS_PATH
PROJECT_ROOT

## Term Patterns

ADHD and Autism are the main target groups. Comparator terms remain separate analysis units. `ASD` is retained only when the local disambiguation window contains autism context, matching the collection-stage rule.

In [ ]:
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)

terms_cfg = config["collection"]["terms"]

term_specs: list[dict[str, object]] = []

def add_term_spec(*, raw_form: str, pattern: str, term_role: str, target_group: str, analysis_unit: str, match_names: set[str] | None = None) -> None:
    term_specs.append(
        {
            "raw_form": raw_form,
            "match_names": match_names or {raw_form},
            "pattern": pattern,
            "term_role": term_role,
            "target_group": target_group,
            "analysis_unit": analysis_unit,
            "regex": re.compile(pattern, re.IGNORECASE),
        }
    )

for spec in terms_cfg["adhd_patterns"]:
    add_term_spec(raw_form=spec["name"], pattern=spec["pattern"], term_role="target", target_group="ADHD", analysis_unit="ADHD")

for spec in terms_cfg["autism_patterns"]:
    add_term_spec(raw_form=spec["name"], pattern=spec["pattern"], term_role="target", target_group="Autism", analysis_unit="Autism")

asd_spec = terms_cfg["asd_pattern"]
add_term_spec(
    raw_form="asd_disambiguated",
    match_names={asd_spec["name"], "asd_disambiguated"},
    pattern=asd_spec["pattern"],
    term_role="target",
    target_group="Autism",
    analysis_unit="Autism",
)

for spec in terms_cfg["baseline_patterns"]:
    add_term_spec(raw_form=spec["name"], pattern=spec["pattern"], term_role="baseline", target_group="baseline", analysis_unit=spec["name"])

term_table = pd.DataFrame([{k: v for k, v in spec.items() if k != "regex"} for spec in term_specs])
term_table

## Context Extraction Helpers

The shared context table needs mention offsets, local target sentences, ±5-token windows, and enough metadata for downstream diagnostics. Overlapping matches within the same analysis unit are resolved by keeping the longest span, so `autism spectrum` is not also counted as a separate `autism` mention at the same offset. Same-analysis-unit acronym/expansion pairs when they occur in the same target sentence and within 120 chars are collapsed, so `attention deficit hyperactivity disorder (ADHD)` now becomes one `ADHD` context row, not two.

In [ ]:
sentence_pattern = re.compile(r"[^.!?]+(?:[.!?]+|$)", re.DOTALL)
token_pattern = re.compile(r"\b\w+(?:[-']\w+)*\b|[^\w\s]", re.UNICODE)

def normalise_space(value: object) -> str:
    return " ".join(str(value or "").split())

def safe_text(value: object) -> str:
    if pd.isna(value):
        return ""
    return str(value)

def safe_int(value: object) -> int | None:
    if pd.isna(value):
        return None
    return int(value)

def split_matched_terms(value: object) -> set[str]:
    if pd.isna(value):
        return set()
    return {part.strip() for part in re.split(r"[|,;]", str(value)) if part.strip()}

def make_doc_id(row_index: int, row: pd.Series) -> str:
    key_parts = [
        str(row_index),
        safe_text(row.get("crawl_id")),
        safe_text(row.get("url")),
        safe_text(row.get("capture_ts")),
        safe_text(row.get("source_year")),
        safe_text(row.get("source_batch")),
        safe_text(row.get("dedup_cluster_id")),
    ]
    return sha1("||".join(key_parts).encode("utf-8")).hexdigest()[:16]

def sentence_spans(excerpt: str) -> list[tuple[int, int, str]]:
    spans: list[tuple[int, int, str]] = []
    for match in sentence_pattern.finditer(excerpt):
        raw_sentence = match.group(0)
        if not raw_sentence.strip():
            continue
        leading_ws = len(raw_sentence) - len(raw_sentence.lstrip())
        trailing_ws = len(raw_sentence.rstrip())
        start = match.start() + leading_ws
        end = match.start() + trailing_ws
        spans.append((start, end, normalise_space(excerpt[start:end])))
    if not spans and excerpt.strip():
        spans.append((0, len(excerpt), normalise_space(excerpt)))
    return spans

def local_sentence_context(text: str, start: int, end: int, window_chars: int = LOCAL_SENTENCE_WINDOW_CHARS) -> tuple[int, str, str]:
    left = max(0, start - window_chars)
    right = min(len(text), end + window_chars)
    excerpt = text[left:right]
    local_start = start - left
    local_end = end - left
    spans = sentence_spans(excerpt)
    if not spans:
        return -1, "", ""
    for idx, (sent_start, sent_end, sent_text) in enumerate(spans):
        if sent_start <= local_start < sent_end or sent_start < local_end <= sent_end:
            adjacent = []
            if idx > 0:
                adjacent.append(spans[idx - 1][2])
            adjacent.append(sent_text)
            if idx + 1 < len(spans):
                adjacent.append(spans[idx + 1][2])
            return idx, sent_text, normalise_space(" ".join(adjacent))
    nearest_idx = min(range(len(spans)), key=lambda i: abs(spans[i][0] - local_start))
    return nearest_idx, spans[nearest_idx][2], spans[nearest_idx][2]

def token_window(text: str, start: int, end: int, token_window_size: int = TOKEN_WINDOW_SIZE, char_window: int = 500) -> str:
    left_char = max(0, start - char_window)
    right_char = min(len(text), end + char_window)
    excerpt = text[left_char:right_char]
    local_start = start - left_char
    local_end = end - left_char
    tokens = list(token_pattern.finditer(excerpt))
    hit_indices = [idx for idx, token in enumerate(tokens) if token.start() < local_end and token.end() > local_start]
    if not hit_indices:
        return normalise_space(excerpt[max(0, local_start - 80): min(len(excerpt), local_end + 80)])
    left = max(0, hit_indices[0] - token_window_size)
    right = min(len(tokens), hit_indices[-1] + token_window_size + 1)
    return normalise_space(excerpt[tokens[left].start(): tokens[right - 1].end()])

def char_context(text: str, start: int, end: int, window_chars: int = SNIPPET_WINDOW_CHARS) -> str:
    return normalise_space(text[max(0, start - window_chars): min(len(text), end + window_chars)])

def asd_is_disambiguated(text: str, start: int, end: int) -> bool:
    left = max(0, start - ASD_DISAMBIGUATION_WINDOW_CHARS)
    right = min(len(text), end + ASD_DISAMBIGUATION_WINDOW_CHARS)
    return "autism" in text[left:right].lower()

def overlaps(left: dict[str, object], right: dict[str, object]) -> bool:
    return int(left["mention_start_char"]) < int(right["mention_end_char"]) and int(right["mention_start_char"]) < int(left["mention_end_char"])

def remove_overlapping_mentions(mentions: list[dict[str, object]]) -> list[dict[str, object]]:
    kept_mentions: list[dict[str, object]] = []
    by_unit: dict[str, list[dict[str, object]]] = defaultdict(list)
    for mention in mentions:
        by_unit[str(mention["analysis_unit"])].append(mention)
    for unit_mentions in by_unit.values():
        selected: list[dict[str, object]] = []
        ordered = sorted(
            unit_mentions,
            key=lambda item: (-(int(item["mention_end_char"]) - int(item["mention_start_char"])), int(item["mention_start_char"]), str(item["raw_form"])),
        )
        for candidate in ordered:
            if any(overlaps(candidate, existing) for existing in selected):
                continue
            selected.append(candidate)
        kept_mentions.extend(sorted(selected, key=lambda item: (int(item["mention_start_char"]), str(item["raw_form"]))))
    return sorted(kept_mentions, key=lambda item: (str(item["analysis_unit"]), int(item["mention_start_char"]), str(item["raw_form"])))


ACRONYM_EXPANSION_GROUPS = [
    {"adhd", "attention_deficit"},
    {"asd_disambiguated", "autism_spectrum"},
]
RAW_FORM_PRIORITY = {
    "attention_deficit": 100,
    "adhd": 10,
    "autism_spectrum": 100,
    "asd_disambiguated": 10,
}

def collapse_group_for_raw_form(raw_form: object) -> set[str] | None:
    raw_form = str(raw_form)
    for group in ACRONYM_EXPANSION_GROUPS:
        if raw_form in group:
            return group
    return None

def mention_gap(left: dict[str, object], right: dict[str, object]) -> int:
    left_start = int(left["mention_start_char"])
    left_end = int(left["mention_end_char"])
    right_start = int(right["mention_start_char"])
    right_end = int(right["mention_end_char"])
    if left_start <= right_end and right_start <= left_end:
        return 0
    return max(right_start - left_end, left_start - right_end)

def can_collapse_mentions(left: dict[str, object], right: dict[str, object], group: set[str]) -> bool:
    if str(left["raw_form"]) not in group or str(right["raw_form"]) not in group:
        return False
    if str(left["raw_form"]) == str(right["raw_form"]):
        return False
    if str(left["analysis_unit"]) != str(right["analysis_unit"]):
        return False
    if str(left["target_sentence"]) != str(right["target_sentence"]):
        return False
    return mention_gap(left, right) <= ACRONYM_EXPANSION_COLLAPSE_WINDOW_CHARS

def annotate_collapsed_mention(cluster: list[dict[str, object]]) -> dict[str, object]:
    primary = max(
        cluster,
        key=lambda item: (
            RAW_FORM_PRIORITY.get(str(item["raw_form"]), 0),
            int(item["mention_end_char"]) - int(item["mention_start_char"]),
            -int(item["mention_start_char"]),
        ),
    ).copy()
    ordered = sorted(cluster, key=lambda item: (int(item["mention_start_char"]), int(item["mention_end_char"]), str(item["raw_form"])))
    primary["collapsed_raw_forms"] = "|".join(sorted({str(item["raw_form"]) for item in ordered}))
    primary["collapsed_matched_texts"] = " | ".join(str(item["matched_text"]) for item in ordered)
    primary["collapsed_match_count"] = len(ordered)
    primary["acronym_expansion_collapsed"] = len({str(item["raw_form"]) for item in ordered}) > 1
    primary["collapsed_mention_start_char"] = min(int(item["mention_start_char"]) for item in ordered)
    primary["collapsed_mention_end_char"] = max(int(item["mention_end_char"]) for item in ordered)
    return primary

def collapse_acronym_expansion_mentions(mentions: list[dict[str, object]]) -> tuple[list[dict[str, object]], int, int]:
    collapsed_mentions: list[dict[str, object]] = []
    collapse_group_count = 0
    mentions_removed = 0
    by_unit: dict[str, list[dict[str, object]]] = defaultdict(list)
    for mention in mentions:
        by_unit[str(mention["analysis_unit"])].append(mention)

    for unit_mentions in by_unit.values():
        remaining = sorted(unit_mentions, key=lambda item: (int(item["mention_start_char"]), int(item["mention_end_char"]), str(item["raw_form"])))
        while remaining:
            seed = remaining.pop(0)
            group = collapse_group_for_raw_form(seed["raw_form"])
            if group is None:
                collapsed_mentions.append(annotate_collapsed_mention([seed]))
                continue

            cluster = [seed]
            changed = True
            while changed:
                changed = False
                next_remaining = []
                for candidate in remaining:
                    if any(can_collapse_mentions(candidate, existing, group) for existing in cluster):
                        cluster.append(candidate)
                        changed = True
                    else:
                        next_remaining.append(candidate)
                remaining = next_remaining

            distinct_raw_forms = {str(item["raw_form"]) for item in cluster}
            if len(distinct_raw_forms) > 1:
                collapse_group_count += 1
                mentions_removed += len(cluster) - 1
            collapsed_mentions.append(annotate_collapsed_mention(cluster))

    return (
        sorted(collapsed_mentions, key=lambda item: (str(item["analysis_unit"]), int(item["mention_start_char"]), str(item["raw_form"]))),
        collapse_group_count,
        mentions_removed,
    )


## Load Corpus

The processed corpus already contains English, deduplicated, quality-gated WARC documents. I use its `matched_terms` metadata to avoid rescanning irrelevant raw forms in each document, while still recovering all mention offsets for the relevant forms.

In [ ]:
columns = [
    "crawl_id",
    "url",
    "registered_domain",
    "capture_ts",
    "published_ts",
    "extracted_text",
    "extracted_text_len",
    "matched_terms",
    "term_roles",
    "dedup_cluster_id",
    "context_snippet_sentence_200",
    "source_corpus_path",
    "source_year",
    "source_batch",
]

corpus = pd.read_parquet(CORPUS_PATH, columns=columns).reset_index(drop=True)
print(f"Loaded {len(corpus):,} processed corpus documents")
corpus[["published_ts", "source_year", "matched_terms", "term_roles", "extracted_text_len"]].head()

## Recover Mention Contexts

Each document can contribute up to three mentions per analysis unit. This cap is applied after overlap resolution, so repeated documents do not dominate annual trajectories while still preserving limited repeated-use signal.

In [ ]:
all_mentions: list[dict[str, object]] = []
documents_with_any_candidate = 0
raw_matches_before_overlap = 0
matches_after_overlap = 0
acronym_expansion_collapse_groups = 0
acronym_expansion_mentions_removed = 0
matches_after_acronym_expansion_collapse = 0
doc_unit_groups_seen = 0
doc_unit_groups_capped = 0

for row_index, row in corpus.iterrows():
    text = safe_text(row.get("extracted_text"))
    if not text.strip():
        continue

    matched_names = split_matched_terms(row.get("matched_terms"))
    lower_text = text.lower()
    candidate_specs = []
    for spec in term_specs:
        if matched_names:
            if not (set(spec["match_names"]) & matched_names):
                continue
        elif str(spec["raw_form"]).split("_")[0] not in lower_text:
            continue
        candidate_specs.append(spec)

    if not candidate_specs:
        continue

    documents_with_any_candidate += 1
    doc_id = make_doc_id(row_index, row)
    doc_mentions: list[dict[str, object]] = []

    for spec in candidate_specs:
        for match in spec["regex"].finditer(text):
            if spec["raw_form"] == "asd_disambiguated" and not asd_is_disambiguated(text, match.start(), match.end()):
                continue
            sentence_index, target_sentence, adjacent_passage = local_sentence_context(text, match.start(), match.end())
            doc_mentions.append(
                {
                    "doc_id": doc_id,
                    "source_row_index": row_index,
                    "crawl_id": safe_text(row.get("crawl_id")),
                    "url": safe_text(row.get("url")),
                    "registered_domain": safe_text(row.get("registered_domain")),
                    "capture_ts": safe_text(row.get("capture_ts")),
                    "published_ts": safe_text(row.get("published_ts")),
                    "source_year": safe_int(row.get("source_year")),
                    "source_batch": safe_int(row.get("source_batch")),
                    "source_corpus_path": safe_text(row.get("source_corpus_path")),
                    "dedup_cluster_id": safe_text(row.get("dedup_cluster_id")),
                    "term_role": spec["term_role"],
                    "target_group": spec["target_group"],
                    "analysis_unit": spec["analysis_unit"],
                    "raw_form": spec["raw_form"],
                    "matched_text": match.group(0),
                    "mention_start_char": match.start(),
                    "mention_end_char": match.end(),
                    "sentence_index_local": sentence_index,
                    "target_sentence": target_sentence,
                    "token_window_5": token_window(text, match.start(), match.end()),
                    "target_sentence_plus_adjacent": adjacent_passage,
                    "context_snippet_sentence_200": char_context(text, match.start(), match.end()),
                    "collection_context_snippet_sentence_200": safe_text(row.get("context_snippet_sentence_200")),
                    "extracted_text_len": safe_int(row.get("extracted_text_len")) or len(text),
                }
            )

    if not doc_mentions:
        continue

    raw_matches_before_overlap += len(doc_mentions)
    doc_mentions = remove_overlapping_mentions(doc_mentions)
    matches_after_overlap += len(doc_mentions)
    doc_mentions, collapse_groups, mentions_removed = collapse_acronym_expansion_mentions(doc_mentions)
    acronym_expansion_collapse_groups += collapse_groups
    acronym_expansion_mentions_removed += mentions_removed
    matches_after_acronym_expansion_collapse += len(doc_mentions)

    by_unit: dict[str, list[dict[str, object]]] = defaultdict(list)
    for mention in doc_mentions:
        by_unit[str(mention["analysis_unit"])].append(mention)

    for unit, unit_mentions in by_unit.items():
        unit_mentions = sorted(unit_mentions, key=lambda item: (int(item["mention_start_char"]), str(item["raw_form"])))
        doc_unit_groups_seen += 1
        if len(unit_mentions) > MENTION_CAP_PER_DOC_UNIT:
            doc_unit_groups_capped += 1
        for mention_index, mention in enumerate(unit_mentions[:MENTION_CAP_PER_DOC_UNIT], start=1):
            mention["mention_index_in_doc_unit"] = mention_index
            mention["matches_before_cap_for_doc_unit"] = len(unit_mentions)
            mention["cap_applied"] = len(unit_mentions) > MENTION_CAP_PER_DOC_UNIT
            all_mentions.append(mention)

contexts = pd.DataFrame(all_mentions)
print(f"Recovered {len(contexts):,} capped mention contexts")
print(f"Documents with candidate terms: {documents_with_any_candidate:,}")
print(f"Raw matches before overlap resolution: {raw_matches_before_overlap:,}")
print(f"Matches after overlap resolution: {matches_after_overlap:,}")
print(f"Acronym/expansion collapse groups: {acronym_expansion_collapse_groups:,}")
print(f"Mentions removed by acronym/expansion collapse: {acronym_expansion_mentions_removed:,}")
contexts.head()

## Save Shared Outputs

The parquet file is the shared handoff table. The CSV files are compact diagnostics for coverage, raw-form balance, domain concentration, and manual inspection.

In [ ]:
if contexts.empty:
    raise RuntimeError("No LSC mention contexts were recovered.")

expected_units = {"ADHD", "Autism", "frustration", "sadness", "loneliness"}
observed_units = set(contexts["analysis_unit"].dropna().unique())
missing_units = sorted(expected_units - observed_units)
if missing_units:
    raise RuntimeError(f"Missing expected analysis units: {missing_units}")

contexts_before_publication_filter = len(contexts)
unique_docs_before_publication_filter = contexts["doc_id"].nunique()
published_dt = pd.to_datetime(contexts["published_ts"].replace("", pd.NA), errors="coerce", utc=True)
capture_dt = pd.to_datetime(contexts["capture_ts"].replace("", pd.NA), errors="coerce", utc=True)
contexts["published_year"] = published_dt.dt.year.astype("Int64")
contexts["capture_year"] = capture_dt.dt.year.astype("Int64")
contexts["lsc_year"] = contexts["published_year"]

missing_or_unparseable_published = contexts["published_year"].isna()
published_before_window = contexts["published_year"].notna() & (contexts["published_year"] < PUBLICATION_YEAR_START)
published_after_window = contexts["published_year"].notna() & (contexts["published_year"] > PUBLICATION_YEAR_END)
kept_publication_window = contexts["published_year"].between(PUBLICATION_YEAR_START, PUBLICATION_YEAR_END)

contexts["publication_date_status"] = "kept_for_lsc_year"
contexts.loc[missing_or_unparseable_published, "publication_date_status"] = "missing_or_unparseable_published_ts"
contexts.loc[published_before_window, "publication_date_status"] = "published_before_lsc_window"
contexts.loc[published_after_window, "publication_date_status"] = "published_after_lsc_window"

publication_exclusions = (
    contexts.groupby(["publication_date_status", "analysis_unit"], dropna=False)
    .agg(rows=("doc_id", "size"), documents=("doc_id", "nunique"), domains=("registered_domain", "nunique"))
    .reset_index()
)
publication_exclusions["publication_year_start"] = PUBLICATION_YEAR_START
publication_exclusions["publication_year_end"] = PUBLICATION_YEAR_END
publication_exclusions.to_csv(PUBLICATION_EXCLUSIONS_PATH, index=False)

contexts = contexts.loc[kept_publication_window].drop(columns=["publication_date_status"]).copy()
contexts["published_year"] = contexts["published_year"].astype(int)
contexts["lsc_year"] = contexts["lsc_year"].astype(int)
if contexts.empty:
    raise RuntimeError("No LSC contexts remain after publication-year filtering.")

observed_units_after_publication_filter = set(contexts["analysis_unit"].dropna().unique())
missing_units_after_publication_filter = sorted(expected_units - observed_units_after_publication_filter)
if missing_units_after_publication_filter:
    raise RuntimeError(f"Missing expected analysis units after publication-year filtering: {missing_units_after_publication_filter}")

max_mentions_per_doc_unit = contexts.groupby(["doc_id", "analysis_unit"]).size().max()
if max_mentions_per_doc_unit > MENTION_CAP_PER_DOC_UNIT:
    raise RuntimeError(f"Mention cap failed: observed max {max_mentions_per_doc_unit}")

contexts = contexts.sort_values(
    ["lsc_year", "analysis_unit", "doc_id", "mention_start_char", "raw_form"]
).reset_index(drop=True)
contexts.to_parquet(CONTEXT_PATH, index=False)

counts_raw = (
    contexts.groupby(["lsc_year", "analysis_unit", "term_role", "target_group", "raw_form"], dropna=False)
    .agg(
        mentions=("doc_id", "size"),
        documents=("doc_id", "nunique"),
        domains=("registered_domain", "nunique"),
        capped_rows=("cap_applied", "sum"),
    )
    .reset_index()
)
counts_raw.to_csv(COUNTS_RAW_PATH, index=False)

counts_unit = (
    contexts.groupby(["lsc_year", "analysis_unit", "term_role", "target_group"], dropna=False)
    .agg(
        mentions=("doc_id", "size"),
        documents=("doc_id", "nunique"),
        domains=("registered_domain", "nunique"),
        raw_forms=("raw_form", "nunique"),
        capped_rows=("cap_applied", "sum"),
    )
    .reset_index()
)
counts_unit.to_csv(COUNTS_UNIT_PATH, index=False)

domain_counts = (
    contexts.groupby(["lsc_year", "analysis_unit", "registered_domain"], dropna=False)
    .agg(mentions=("doc_id", "size"), documents=("doc_id", "nunique"))
    .reset_index()
    .sort_values(["lsc_year", "analysis_unit", "mentions", "registered_domain"], ascending=[True, True, False, True])
)
domain_counts["total_mentions_for_year_unit"] = domain_counts.groupby(["lsc_year", "analysis_unit"])["mentions"].transform("sum")
domain_counts["mention_share"] = domain_counts["mentions"] / domain_counts["total_mentions_for_year_unit"]
domain_counts["domain_rank"] = domain_counts.groupby(["lsc_year", "analysis_unit"]).cumcount() + 1
top_domains = domain_counts.loc[domain_counts["domain_rank"] <= 10].copy()
top_domains.to_csv(TOP_DOMAINS_PATH, index=False)

sample_frames = []
for _, frame in contexts.groupby(["analysis_unit", "lsc_year"], sort=True):
    sample_frames.append(frame.sample(n=min(3, len(frame)), random_state=RANDOM_SEED))
samples = pd.concat(sample_frames, ignore_index=True).sort_values(["analysis_unit", "lsc_year", "raw_form", "doc_id"])
sample_columns = [
    "lsc_year",
    "published_year",
    "source_year",
    "analysis_unit",
    "raw_form",
    "matched_text",
    "collapsed_raw_forms",
    "collapsed_matched_texts",
    "collapsed_match_count",
    "acronym_expansion_collapsed",
    "registered_domain",
    "url",
    "target_sentence",
    "token_window_5",
    "target_sentence_plus_adjacent",
]
samples[sample_columns].to_csv(SAMPLES_PATH, index=False)

summary = pd.DataFrame(
    [
        {"metric": "run_completed_utc", "value": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")},
        {"metric": "source_documents", "value": len(corpus)},
        {"metric": "documents_with_candidate_terms", "value": documents_with_any_candidate},
        {"metric": "raw_matches_before_overlap_resolution", "value": raw_matches_before_overlap},
        {"metric": "matches_after_overlap_resolution", "value": matches_after_overlap},
        {"metric": "acronym_expansion_collapse_groups", "value": acronym_expansion_collapse_groups},
        {"metric": "acronym_expansion_mentions_removed", "value": acronym_expansion_mentions_removed},
        {"metric": "matches_after_acronym_expansion_collapse", "value": matches_after_acronym_expansion_collapse},
        {"metric": "mention_context_rows_before_publication_filter", "value": contexts_before_publication_filter},
        {"metric": "unique_documents_before_publication_filter", "value": unique_docs_before_publication_filter},
        {"metric": "missing_or_unparseable_published_ts_rows", "value": int(missing_or_unparseable_published.sum())},
        {"metric": "published_before_lsc_window_rows", "value": int(published_before_window.sum())},
        {"metric": "published_after_lsc_window_rows", "value": int(published_after_window.sum())},
        {"metric": "publication_year_start", "value": PUBLICATION_YEAR_START},
        {"metric": "publication_year_end", "value": PUBLICATION_YEAR_END},
        {"metric": "mention_context_rows_after_cap", "value": len(contexts)},
        {"metric": "unique_documents_with_context", "value": contexts["doc_id"].nunique()},
        {"metric": "unique_domains_with_context", "value": contexts["registered_domain"].nunique()},
        {"metric": "mention_cap_per_doc_analysis_unit", "value": MENTION_CAP_PER_DOC_UNIT},
        {"metric": "doc_analysis_unit_groups_seen", "value": doc_unit_groups_seen},
        {"metric": "doc_analysis_unit_groups_capped", "value": doc_unit_groups_capped},
        {"metric": "rows_from_capped_doc_units", "value": int(contexts["cap_applied"].sum())},
    ]
)
summary.to_csv(SUMMARY_PATH, index=False)

print("Wrote shared LSC context outputs:")
for path in [CONTEXT_PATH, COUNTS_RAW_PATH, COUNTS_UNIT_PATH, TOP_DOMAINS_PATH, SAMPLES_PATH, SUMMARY_PATH, PUBLICATION_EXCLUSIONS_PATH]:
    print(f"- {path.relative_to(PROJECT_ROOT)}")

counts_unit.head()